
# 03 — NLP Sentiment per Ulasan (Final)

Notebook ini menjalankan sentiment analysis terhadap **setiap ulasan secara individual**, kemudian mengagregasikan hasilnya menjadi satu sentimen untuk setiap listing usaha.

## Input resmi

`03_Input_NLP_Ringkas.csv`

Kolom input:

- `entity_key` — ID teknis untuk menjaga identitas baris; bukan fitur NLP atau K-Means.
- `title`
- `totalScore`
- `reviewsCount`
- `street`
- `city`
- `categoryName`
- `text`
- `jumlah_teks_untuk_sentimen`

## Model

`w11wo/indonesian-roberta-base-sentiment-classifier`

Label model:

- `positive`
- `neutral`
- `negative`

## Perbaikan metodologis

Versi sebelumnya memprediksi teks gabungan dan memotongnya berdasarkan karakter. Notebook ini:

1. Memisahkan kembali teks gabungan menggunakan separator `|||`.
2. Memprediksi setiap ulasan secara individual.
3. Menyimpan probabilitas setiap label.
4. Menentukan sentimen usaha menggunakan suara mayoritas.
5. Menyimpan proporsi sentimen sebagai bukti audit.
6. Menghasilkan file modeling yang ringkas.

## Arti `sentimen_confidence`

Kolom `sentimen_confidence` **bukan probabilitas keyakinan model**. Kolom ini hanya menunjukkan kecukupan jumlah teks:

- `Rendah` = 1 ulasan
- `Sedang` = 2 ulasan
- `Tinggi` = minimal 3 ulasan

Probabilitas model disimpan terpisah dalam file audit sebagai `model_confidence_mean`.


In [ ]:

# ============================================================
# 1. INSTALASI LIBRARY
# ============================================================
!pip install -q transformers accelerate openpyxl


In [ ]:

# ============================================================
# 2. IMPORT LIBRARY
# ============================================================
import os
import re
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForSequenceClassification

try:
    from google.colab import files
    RUNNING_IN_COLAB = True
except ImportError:
    RUNNING_IN_COLAB = False

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 150)

print("PyTorch:", torch.__version__)
print("CUDA tersedia:", torch.cuda.is_available())


In [ ]:

# ============================================================
# 3. KONFIGURASI
# ============================================================
INPUT_FILE = "03_Input_NLP_Ringkas.csv"

MODEL_NAME = "w11wo/indonesian-roberta-base-sentiment-classifier"
REVIEW_SEPARATOR_PATTERN = r"\s*\|\|\|\s*"

OUTPUT_INPUT_NLP = "03_Input_NLP_Ringkas_Tervalidasi.csv"
OUTPUT_CHECKPOINT = "03_Prediksi_Review_Checkpoint.csv"
OUTPUT_AUDIT_CSV = "03_Hasil_NLP_Audit.csv"
OUTPUT_MODELING_CSV = "03_Data_Modeling_Setelah_NLP.csv"

MAX_LENGTH = 512
BATCH_SIZE_GPU = 32
BATCH_SIZE_CPU = 8
CHECKPOINT_EVERY = 1000

EXPECTED_ENTITY_COUNT = 3438

LABEL_TO_SCORE = {
    "positive": 1,
    "neutral": 0,
    "negative": -1,
}

print("Konfigurasi selesai.")


In [ ]:

# ============================================================
# 4. UPLOAD INPUT
# ============================================================
if not os.path.exists(INPUT_FILE):
    if not RUNNING_IN_COLAB:
        raise FileNotFoundError(INPUT_FILE)

    print("Upload file:", INPUT_FILE)
    uploaded = files.upload()

    if INPUT_FILE not in uploaded:
        csv_files = [
            name for name in uploaded
            if name.lower().endswith(".csv")
        ]

        if len(csv_files) != 1:
            raise FileNotFoundError(
                "Upload tepat satu file CSV input NLP."
            )

        os.rename(csv_files[0], INPUT_FILE)

print("File input ditemukan:", INPUT_FILE)


In [ ]:

# ============================================================
# 5. MEMBACA DAN MEMVALIDASI INPUT NLP
# ============================================================
df = pd.read_csv(
    INPUT_FILE,
    sep=";",
    encoding="utf-8-sig",
    decimal=",",
    on_bad_lines="warn",
)

required_columns = [
    "entity_key",
    "title",
    "totalScore",
    "reviewsCount",
    "street",
    "city",
    "categoryName",
    "text",
    "jumlah_teks_untuk_sentimen",
]

missing_columns = [
    column for column in required_columns
    if column not in df.columns
]

if missing_columns:
    raise ValueError(
        f"Kolom input NLP tidak lengkap: {missing_columns}"
    )

df = df[required_columns].copy()

df["totalScore"] = pd.to_numeric(
    df["totalScore"]
    .astype(str)
    .str.replace(",", ".", regex=False),
    errors="coerce",
)

df["reviewsCount"] = pd.to_numeric(
    df["reviewsCount"],
    errors="coerce",
)

df["jumlah_teks_untuk_sentimen"] = pd.to_numeric(
    df["jumlah_teks_untuk_sentimen"],
    errors="coerce",
).astype("Int64")

assert len(df) == EXPECTED_ENTITY_COUNT, (
    f"Jumlah data bukan {EXPECTED_ENTITY_COUNT}: {len(df)}"
)

assert df["entity_key"].notna().all()
assert df["entity_key"].is_unique
assert df["title"].notna().all()
assert df["text"].notna().all()
assert df["totalScore"].between(
    1, 5, inclusive="both"
).all()
assert (df["reviewsCount"] >= 1).all()

print("Jumlah entitas:", len(df))
print("Jumlah kolom  :", df.shape[1])
display(df.head(3))


In [ ]:

# ============================================================
# 6. FUNGSI PEMBERSIHAN DAN PEMISAHAN ULASAN
# ============================================================
INVALID_TEXT_VALUES = {
    "",
    "nan",
    "none",
    "null",
    "#name?",
    "-",
    "tidak ada ulasan",
}

def clean_text(value):
    if pd.isna(value):
        return ""

    value = unicodedata.normalize(
        "NFKC",
        str(value),
    )

    value = re.sub(
        r"\s+",
        " ",
        value,
    ).strip()

    return value


def normalize_text(value):
    return clean_text(value).lower()


def split_reviews(value):
    value = clean_text(value)

    parts = re.split(
        REVIEW_SEPARATOR_PATTERN,
        value,
    )

    unique_reviews = []
    seen = set()

    for part in parts:
        original = clean_text(part)
        normalized = normalize_text(original)

        if normalized in INVALID_TEXT_VALUES:
            continue

        if normalized not in seen:
            seen.add(normalized)
            unique_reviews.append(original)

    return unique_reviews

df["_review_list"] = df["text"].apply(split_reviews)
df["_jumlah_teks_hitung"] = df["_review_list"].apply(len)

count_mismatch = df[
    df["_jumlah_teks_hitung"]
    != df["jumlah_teks_untuk_sentimen"]
].copy()

print("Perbedaan jumlah teks:", len(count_mismatch))

if len(count_mismatch) > 0:
    display(
        count_mismatch[
            [
                "entity_key",
                "title",
                "jumlah_teks_untuk_sentimen",
                "_jumlah_teks_hitung",
            ]
        ].head(20)
    )
    raise ValueError(
        "Jumlah teks pada kolom tidak sesuai hasil pemisahan."
    )

print(
    "Total ulasan individual:",
    int(df["_jumlah_teks_hitung"].sum()),
)


In [ ]:

# ============================================================
# 7. MEMBENTUK DATA PER ULASAN
# ============================================================
review_records = []

for _, row in df.iterrows():
    for review_number, review_text in enumerate(
        row["_review_list"],
        start=1,
    ):
        review_records.append({
            "review_key": (
                f"{row['entity_key']}::"
                f"{review_number}"
            ),
            "entity_key": row["entity_key"],
            "title": row["title"],
            "review_number": review_number,
            "review_text": review_text,
        })

df_reviews = pd.DataFrame(review_records)

assert df_reviews["review_key"].is_unique
assert df_reviews["review_text"].notna().all()
assert df_reviews["review_text"].str.strip().ne("").all()

print("Jumlah entitas       :", df["entity_key"].nunique())
print("Jumlah ulasan NLP    :", len(df_reviews))
print("Rata-rata per entitas:", len(df_reviews) / len(df))

display(df_reviews.head(10))


In [ ]:

# ============================================================
# 8. MEMUAT MODEL INDO-ROBERTA
# ============================================================
device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

batch_size = (
    BATCH_SIZE_GPU
    if torch.cuda.is_available()
    else BATCH_SIZE_CPU
)

print("Device     :", device)
print("Batch size :", batch_size)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME
)

model.to(device)
model.eval()

id2label = {
    int(index): str(label).lower()
    for index, label
    in model.config.id2label.items()
}

expected_fallback = {
    0: "positive",
    1: "neutral",
    2: "negative",
}

for index, label in list(id2label.items()):
    if label.startswith("label_"):
        id2label[index] = expected_fallback[index]

print("Pemetaan label model:", id2label)

assert set(id2label.values()) == {
    "positive",
    "neutral",
    "negative",
}, "Label model tidak sesuai tiga kelas yang diharapkan."



## Catatan checkpoint

Prediksi sekitar 14 ribu ulasan dapat memerlukan waktu. Notebook menyimpan checkpoint secara berkala ke:

`03_Prediksi_Review_Checkpoint.csv`

Apabila runtime terputus:

1. Jalankan kembali notebook.
2. Upload file input NLP.
3. Upload checkpoint bila tersedia.
4. Notebook akan melewati review yang sudah diprediksi.


In [ ]:

# ============================================================
# 9. MEMUAT CHECKPOINT JIKA ADA
# ============================================================
prediction_columns = [
    "review_key",
    "entity_key",
    "title",
    "review_number",
    "review_text",
    "predicted_label",
    "predicted_score",
    "prob_positive",
    "prob_neutral",
    "prob_negative",
    "was_truncated",
]

if os.path.exists(OUTPUT_CHECKPOINT):
    checkpoint = pd.read_csv(
        OUTPUT_CHECKPOINT,
        sep=";",
        encoding="utf-8-sig",
        decimal=",",
    )

    checkpoint = checkpoint[
        checkpoint["review_key"].isin(
            df_reviews["review_key"]
        )
    ].copy()

    checkpoint = checkpoint.drop_duplicates(
        "review_key",
        keep="last",
    )

    print(
        "Checkpoint ditemukan:",
        len(checkpoint),
        "prediksi",
    )
else:
    checkpoint = pd.DataFrame(
        columns=prediction_columns
    )
    print("Belum ada checkpoint.")

predicted_keys = set(checkpoint["review_key"])

df_pending = df_reviews[
    ~df_reviews["review_key"].isin(
        predicted_keys
    )
].copy()

print("Sudah diprediksi :", len(checkpoint))
print("Belum diprediksi :", len(df_pending))


In [ ]:

# ============================================================
# 10. PREDIKSI SENTIMEN PER ULASAN
# ============================================================
new_predictions = []

pending_records = df_pending.to_dict("records")

for start in tqdm(
    range(0, len(pending_records), batch_size),
    desc="Prediksi sentimen",
):
    batch_records = pending_records[
        start:start + batch_size
    ]

    batch_texts = [
        record["review_text"]
        for record in batch_records
    ]

    lengths_without_truncation = tokenizer(
        batch_texts,
        add_special_tokens=True,
        truncation=False,
        padding=False,
    )["input_ids"]

    was_truncated_flags = [
        len(token_ids) > MAX_LENGTH
        for token_ids in lengths_without_truncation
    ]

    encoded = tokenizer(
        batch_texts,
        padding=True,
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors="pt",
    )

    encoded = {
        key: value.to(device)
        for key, value in encoded.items()
    }

    with torch.inference_mode():
        logits = model(**encoded).logits

        probabilities = torch.softmax(
            logits,
            dim=-1,
        ).cpu().numpy()

    for record, prob_values, was_truncated in zip(
        batch_records,
        probabilities,
        was_truncated_flags,
    ):
        probability_by_label = {
            id2label[index]: float(
                prob_values[index]
            )
            for index in range(
                len(prob_values)
            )
        }

        predicted_label = max(
            probability_by_label,
            key=probability_by_label.get,
        )

        new_predictions.append({
            **record,
            "predicted_label": predicted_label,
            "predicted_score": (
                probability_by_label[
                    predicted_label
                ]
            ),
            "prob_positive": (
                probability_by_label[
                    "positive"
                ]
            ),
            "prob_neutral": (
                probability_by_label[
                    "neutral"
                ]
            ),
            "prob_negative": (
                probability_by_label[
                    "negative"
                ]
            ),
            "was_truncated": was_truncated,
        })

    if (
        len(new_predictions) >= CHECKPOINT_EVERY
        or start + batch_size >= len(
            pending_records
        )
    ):
        new_df = pd.DataFrame(
            new_predictions
        )

        checkpoint = pd.concat(
            [checkpoint, new_df],
            ignore_index=True,
        )

        checkpoint = checkpoint.drop_duplicates(
            "review_key",
            keep="last",
        )

        checkpoint[prediction_columns].to_csv(
            OUTPUT_CHECKPOINT,
            index=False,
            sep=";",
            encoding="utf-8-sig",
            decimal=",",
        )

        new_predictions = []

        print(
            "Checkpoint tersimpan:",
            len(checkpoint),
            "/",
            len(df_reviews),
        )

df_predictions = checkpoint[
    prediction_columns
].copy()

assert len(df_predictions) == len(df_reviews)
assert df_predictions["review_key"].is_unique
assert df_predictions["predicted_label"].isin(
    LABEL_TO_SCORE
).all()

print("Semua review berhasil diprediksi:", len(df_predictions))


In [ ]:

# ============================================================
# 11. AGREGASI SENTIMEN PER ENTITAS
# ============================================================
label_order = [
    "positive",
    "neutral",
    "negative",
]

count_table = pd.crosstab(
    df_predictions["entity_key"],
    df_predictions["predicted_label"],
).reindex(
    columns=label_order,
    fill_value=0,
)

count_table.columns = [
    f"jumlah_{label}"
    for label in count_table.columns
]

probability_mean = (
    df_predictions
    .groupby("entity_key")[
        [
            "prob_positive",
            "prob_neutral",
            "prob_negative",
            "predicted_score",
        ]
    ]
    .mean()
    .rename(
        columns={
            "predicted_score": (
                "model_confidence_mean"
            ),
        }
    )
)

review_count_prediction = (
    df_predictions
    .groupby("entity_key")
    .size()
    .rename("jumlah_prediksi_review")
)

aggregation = (
    count_table
    .join(probability_mean)
    .join(review_count_prediction)
    .reset_index()
)

for label in label_order:
    aggregation[f"proporsi_{label}"] = (
        aggregation[f"jumlah_{label}"]
        / aggregation[
            "jumlah_prediksi_review"
        ]
    )


def choose_majority_label(row):
    counts = {
        label: int(
            row[f"jumlah_{label}"]
        )
        for label in label_order
    }

    maximum_count = max(counts.values())

    candidates = [
        label
        for label, count in counts.items()
        if count == maximum_count
    ]

    if len(candidates) == 1:
        return candidates[0]

    return max(
        candidates,
        key=lambda label: row[
            f"prob_{label}"
        ],
    )


aggregation["sentiment_label"] = aggregation.apply(
    choose_majority_label,
    axis=1,
)

aggregation["sentiment_score"] = aggregation[
    "sentiment_label"
].map(LABEL_TO_SCORE)

print("Agregasi selesai.")
display(aggregation.head())


In [ ]:

# ============================================================
# 12. MEMBENTUK DATASET SETELAH NLP
# ============================================================
df_result = df.drop(
    columns=[
        "_review_list",
        "_jumlah_teks_hitung",
    ],
).merge(
    aggregation,
    on="entity_key",
    how="left",
    validate="one_to_one",
)

df_result["sentimen_confidence"] = np.select(
    [
        df_result[
            "jumlah_teks_untuk_sentimen"
        ] >= 3,
        df_result[
            "jumlah_teks_untuk_sentimen"
        ] == 2,
        df_result[
            "jumlah_teks_untuk_sentimen"
        ] == 1,
    ],
    [
        "Tinggi",
        "Sedang",
        "Rendah",
    ],
    default="Tidak valid",
)

modeling_columns = [
    "entity_key",
    "title",
    "totalScore",
    "reviewsCount",
    "street",
    "city",
    "categoryName",
    "text",
    "jumlah_teks_untuk_sentimen",
    "sentiment_score",
    "sentimen_confidence",
]

df_modeling = df_result[
    modeling_columns
].copy()

assert len(df_modeling) == EXPECTED_ENTITY_COUNT
assert df_modeling["entity_key"].is_unique
assert df_modeling["sentiment_score"].isin(
    [-1, 0, 1]
).all()
assert df_modeling["sentimen_confidence"].isin(
    ["Tinggi", "Sedang", "Rendah"]
).all()

print("Ukuran dataset modeling:", df_modeling.shape)
display(df_modeling.head(3))


In [ ]:

# ============================================================
# 13. RINGKASAN DAN REKONSILIASI
# ============================================================
ringkasan_sentimen = (
    df_result["sentiment_label"]
    .value_counts()
    .reindex(
        ["positive", "neutral", "negative"],
        fill_value=0,
    )
    .rename_axis("sentiment_label")
    .reset_index(name="jumlah")
)

ringkasan_sentimen["persentase"] = (
    ringkasan_sentimen["jumlah"]
    / len(df_result)
    * 100
)

ringkasan_kecukupan = (
    df_result["sentimen_confidence"]
    .value_counts()
    .reindex(
        ["Tinggi", "Sedang", "Rendah"],
        fill_value=0,
    )
    .rename_axis("sentimen_confidence")
    .reset_index(name="jumlah")
)

ringkasan_kecukupan["persentase"] = (
    ringkasan_kecukupan["jumlah"]
    / len(df_result)
    * 100
)

rekonsiliasi = pd.DataFrame([
    {
        "tahap": "Input NLP",
        "jumlah_entitas": len(df),
        "jumlah_ulasan_individual": len(df_reviews),
        "keterangan": (
            "Dataset final setelah merger dan "
            "deduplikasi lintas anggota."
        ),
    },
    {
        "tahap": "Prediksi per ulasan",
        "jumlah_entitas": (
            df_predictions[
                "entity_key"
            ].nunique()
        ),
        "jumlah_ulasan_individual": len(
            df_predictions
        ),
        "keterangan": (
            "Seluruh ulasan diprediksi menggunakan "
            "Indo-RoBERTa."
        ),
    },
    {
        "tahap": "Agregasi per entitas",
        "jumlah_entitas": len(df_modeling),
        "jumlah_ulasan_individual": len(
            df_predictions
        ),
        "keterangan": (
            "Label mayoritas dipetakan menjadi "
            "sentiment_score -1, 0, atau 1."
        ),
    },
])

display(ringkasan_sentimen)
display(ringkasan_kecukupan)
display(rekonsiliasi)


In [ ]:
# ============================================================
# 14. EKSPOR OUTPUT
# ============================================================
df[[
    "entity_key",
    "title",
    "totalScore",
    "reviewsCount",
    "street",
    "city",
    "categoryName",
    "text",
    "jumlah_teks_untuk_sentimen",
]].to_csv(
    OUTPUT_INPUT_NLP,
    index=False,
    sep=";",
    encoding="utf-8-sig"
)

df_modeling.to_csv(
    OUTPUT_MODELING_CSV,
    index=False,
    sep=";",
    encoding="utf-8-sig"
)

df_result[audit_entity_columns].to_csv(
    OUTPUT_AUDIT_CSV,
    index=False,
    sep=";",
    encoding="utf-8-sig"
)

df_predictions.to_csv(
    "03_Hasil_NLP_Prediksi_Per_Ulasan.csv",
    index=False,
    sep=";",
    encoding="utf-8-sig"
)

ringkasan_sentimen.to_csv(
    "03_Hasil_NLP_Ringkasan_Sentimen.csv",
    index=False,
    sep=";",
    encoding="utf-8-sig"
)

print("Berhasil mengespor:")
print("-", OUTPUT_MODELING_CSV)
print("-", OUTPUT_AUDIT_CSV)
print("-", OUTPUT_CHECKPOINT)


In [ ]:

# ============================================================
# 15. PEMERIKSAAN AKHIR WAJIB
# ============================================================
assert len(df_modeling) == EXPECTED_ENTITY_COUNT
assert df_modeling["entity_key"].nunique() == EXPECTED_ENTITY_COUNT
assert df_modeling["sentiment_score"].notna().all()
assert (
    df_result["jumlah_prediksi_review"]
    == df_result[
        "jumlah_teks_untuk_sentimen"
    ]
).all()

print("SEMUA VALIDASI NLP LULUS.")
print("Jumlah final modeling:", len(df_modeling))

print("\nKolom file modeling:")
for column in df_modeling.columns:
    print("-", column)

print(
    "\nFitur K-Means nanti hanya:"
    "\n- totalScore"
    "\n- log_reviewsCount"
    "\n- sentiment_score"
)


In [ ]:
# ============================================================
# 16. DOWNLOAD
# ============================================================
if RUNNING_IN_COLAB:
    files.download(OUTPUT_MODELING_CSV)
    files.download(OUTPUT_AUDIT_CSV)
    files.download(OUTPUT_CHECKPOINT)
else:
    print("File tersimpan pada folder kerja.")



## File untuk tahap selanjutnya

Gunakan:

`03_Data_Modeling_Setelah_NLP.csv`

sebagai input notebook clustering.

Kolom berikut **bukan fitur clustering**, tetapi dipertahankan untuk identifikasi, profiling, dan rekomendasi:

- `entity_key`
- `title`
- `street`
- `city`
- `categoryName`
- `text`
- `jumlah_teks_untuk_sentimen`
- `sentimen_confidence`

Fitur K-Means hanya:

1. `totalScore`
2. `log_reviewsCount`, yang dibentuk dari `reviewsCount`
3. `sentiment_score`
